## Setting things up!

In [44]:
import glob
import re
import json
import os
import random
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

In [45]:
FOLDER = str(Path("treebanks") / "perseus")

# Directory containing modular grammar lesson files
GRAMMAR_FOLDER = Path("lessons-no-decl")


In [46]:
folder = Path(FOLDER)

def clean_text(element):
    return " ".join(element.itertext()).split() if element is not None else []

rows = []
for xml_file in sorted(folder.glob("*.xml")):
    root = ET.parse(xml_file).getroot()
    title = " ".join(clean_text(root.find(".//title"))) or None
    author = " ".join(clean_text(root.find(".//author"))) or None

    rows.append({
        "file": xml_file.name,
        "title": title,
        "author": author,
    })

Available_treebanks = pd.DataFrame(rows)
#print(Available_treebanks)


In [47]:
# interactive treebank selector (only sets CHOSEN_TEXTS)
try:
    import ipywidgets as widgets
    from IPython.display import display

    # Show available treebanks before asking for row input.
    if 'Available_treebanks' in globals() and not Available_treebanks.empty:
        options_preview = Available_treebanks.reset_index().rename(columns={'index': 'option'})
        print("Available treebanks (choose by option index):")
        print(options_preview[['option', 'file', 'author', 'title']].to_string(index=False))
    else:
        print("Available_treebanks is missing or empty. Run the previous listing cell first.")

    selector = widgets.Text(
        value="all",
        description="Rows:",
        placeholder="all or 0,2,5-8",
        layout=widgets.Layout(width="420px"),
    )
    apply_button = widgets.Button(description="Apply", button_style="primary")
    status = widgets.HTML("")

    def _apply_selection(_):
        global CHOSEN_TEXTS
        CHOSEN_TEXTS = (selector.value or "").strip() or "all"
        status.value = f"Selection saved to <code>CHOSEN_TEXTS</code>: <b>{CHOSEN_TEXTS}</b>"

    apply_button.on_click(_apply_selection)

    display(widgets.HBox([selector, apply_button]))
    display(status)
    print("Enter all or row indices (e.g., 0,2,5-8), click Apply, then run the next cell.")

except Exception:
    if 'Available_treebanks' in globals() and not Available_treebanks.empty:
        options_preview = Available_treebanks.reset_index().rename(columns={'index': 'option'})
        print("Available treebanks (choose by option index):")
        print(options_preview[['option', 'file', 'author', 'title']].to_string(index=False))
    else:
        print("Available_treebanks is missing or empty. Run the previous listing cell first.")

    CHOSEN_TEXTS = input("Select rows to include (all or 0,2,5-8): ").strip() or "all"
    print(f"Selection saved to CHOSEN_TEXTS: {CHOSEN_TEXTS}")


Available treebanks (choose by option index):
 option                                           file           author                                                                                                                                      title
      0           tlg0003.tlg001.perseus-grc1.1.tb.xml       Thucydides                                                                                                                   Historiae in two volumes
      1             tlg0007.tlg004.perseus-grc1.tb.xml         Plutarch                                                                                                                           Plutarch's Lives
      2             tlg0007.tlg015.perseus-grc1.tb.xml         Plutarch                                                                                                                           Plutarch's Lives
      3          tlg0008.tlg001.perseus-grc1.12.tb.xml        Athenaeus                                       

HTML(value='')

Enter all or row indices (e.g., 0,2,5-8), click Apply, then run the next cell.


## Treebank XML to DF

In [48]:
# Parse a treebank XML file and return a pandas DataFrame.
def parse_treebank_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()

    end_punct = {'.', '?', ';', '!', ':'}
    data, sentence_counter, token_index = [], 1, 0
    current_sentence_id = f"tb_{sentence_counter}"

    for sentence in root.findall('.//sentence'):
        document_id = sentence.get('document_id')
        subdoc = sentence.get('subdoc')

        for word in sentence.findall('word'):
            token_index += 1
            form = word.get('form') or ''

            data.append({
                'sentence_id': current_sentence_id,
                'document_id': document_id,
                'subdoc': subdoc,
                'word_id': word.get('id'),
                'token_index': token_index,
                'form': form,
                'lemma': word.get('lemma'),
                'postag': word.get('postag'),
                'relation': word.get('relation'),
                'head': word.get('head')
            })

            if form in end_punct:
                sentence_counter += 1
                current_sentence_id = f"tb_{sentence_counter}"

    return pd.DataFrame(data)


def parse_row_selection(selection_text, max_index):
    text = str(selection_text).strip().lower()
    if text in {'', 'all', '*'}:
        return list(range(max_index + 1))

    picked = set()
    for part in text.split(','):
        part = part.strip()
        if not part:
            continue
        if '-' in part:
            a, b = map(str.strip, part.split('-', 1))
            lo, hi = sorted((int(a), int(b)))
            picked.update(i for i in range(lo, hi + 1) if 0 <= i <= max_index)
        else:
            i = int(part)
            if 0 <= i <= max_index:
                picked.add(i)
    return sorted(picked)


if 'Available_treebanks' not in globals() or Available_treebanks.empty:
    raise ValueError("Available_treebanks is missing or empty. Run the treebank-listing cell first.")

options_df = Available_treebanks.reset_index().rename(columns={'index': 'option'})

chosen_text = str(globals().get('CHOSEN_TEXTS', '')).strip().lower()
chosen_rows = parse_row_selection(chosen_text, len(options_df) - 1)
selected_treebanks = options_df[options_df['option'].isin(chosen_rows)].copy()

if selected_treebanks.empty:
    raise ValueError("No valid treebanks were selected.")

xml_files = [os.path.join(FOLDER, f) for f in selected_treebanks['file']]
print(f"Selected {len(xml_files)} treebank file(s).")
print(selected_treebanks[['file', 'author', 'title']].to_string(index=False))

all_dfs = []
for file_path in xml_files:
    df = parse_treebank_xml(file_path)
    df['file'] = os.path.basename(file_path)
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df.sample(5)


Selected 16 treebank file(s).
                                 file           author                                                                                                                                      title
tlg0008.tlg001.perseus-grc1.12.tb.xml        Athenaeus                                                                                                                         The Deipnosophists
tlg0008.tlg001.perseus-grc1.13.tb.xml        Athenaeus                                                                                                                         The Deipnosophists
   tlg0011.tlg001.perseus-grc2.tb.xml        Sophocles          Sophocles. Vol 2: Ajax. Electra. Trachiniae. Philoctetes With an English translation by F. Storr. The Loeb classical library, 21.
   tlg0011.tlg002.perseus-grc2.tb.xml        Sophocles Sophocles. Vol 1: Oedipus the king. Oedipus at Colonus. Antigone. With an English translation by F. Storr. The Loeb classical library, 20.


,sentence_id,document_id,subdoc,word_id,token_index,form,lemma,postag,relation,head,file
48491,tb_151,urn:cts:greekLit:tlg0011.tlg001.perseus-grc2,403,1,2907,σὺ,σύ,p-s---mn-,SBJ,8,tlg0011.tlg001.perseus-grc2.tb.xml
240864,tb_492,urn:cts:greekLit:tlg0012.tlg002.perseus-grc1,12.39-12.40,11,14705,θέλγουσιν,θέλγω,v3ppia---,ATR,1,tlg0012.tlg002.perseus-grc1.tb.xml
93986,tb_541,urn:cts:greekLit:tlg0011.tlg005.perseus-grc2,1288-1291,18,9114,πατρῴαν,πατρῷος,a-s---fa-,ATR,19,tlg0011.tlg005.perseus-grc2.tb.xml
110975,tb_438,urn:cts:greekLit:tlg0012.tlg001.perseus-grc1,11.618-11.621,24,15297,ἐξ,ἐκ,r--------,AuxP,21,tlg0012.tlg001.perseus-grc1.tb.xml
21022,tb_24,urn:cts:greekLit:tlg0008.tlg001.perseus-grc1,13.4,7,727,αὐτῷ,αὐτός,p-s---md-,ATR,8,tlg0008.tlg001.perseus-grc1.13.tb.xml


## Parsing POS tags for the frequency-based syllabus

In [49]:
import unicodedata

# POS decoding maps
CASE_MAP = {"n": "nominative", "g": "genitive", "d": "dative", "a": "accusative", "v": "vocative"}
TENSE_MAP = {"p": "present", "i": "imperfect", "f": "future", "a": "aorist", "r": "perfect", "l": "pluperfect", "t": "future perfect"}
MOOD_MAP = {"i": "indicative", "s": "subjunctive", "o": "optative", "m": "imperative", "n": "infinitive", "p": "participle"}
VOICE_MAP = {"a": "active", "m": "middle", "p": "passive", "e": "middle/passive"}

# Simple POS classes stay one lesson each in the syllabus.
SIMPLE_POS_LABELS = {
    "d": "adverb",
    "r": "preposition",
    "g": "particle",
    "c": "conjunction",
    "i": "interjection",
}

POS_CATEGORY_MAP = {
    "v": "verb",
    "l": "article",
    "p": "pronoun",
    **SIMPLE_POS_LABELS,
}


def _decode(code_map, ch):
    return "unknown" if ch == "-" else code_map.get(ch, ch)


def parse_postag(postag):
    if not isinstance(postag, str) or not postag:
        return "NA"

    pos = postag[0]

    if pos in {"n", "a"} and len(postag) > 7:
        return _decode(CASE_MAP, postag[7])

    if pos == "v" and len(postag) > 5:
        return ", ".join([
            _decode(TENSE_MAP, postag[3]),
            _decode(MOOD_MAP, postag[4]),
            _decode(VOICE_MAP, postag[5]),
        ])

    if pos == "l":
        return "article"
    if pos == "p":
        return "pronoun"
    if pos in SIMPLE_POS_LABELS:
        return SIMPLE_POS_LABELS[pos]
    return "NA"


def parse_pos_category(postag):
    if not isinstance(postag, str) or not postag:
        return "other"
    return POS_CATEGORY_MAP.get(postag[0], "noun/adjective" if postag[0] in {"n", "a"} else "other")


def normalize_greek_lemma(lemma):
    if not isinstance(lemma, str):
        return ""
    return "".join(
        c for c in unicodedata.normalize("NFD", lemma.lower().strip())
        if unicodedata.category(c) != "Mn"
    )


def parse_verb_subcategory(lemma, postag=None):
    if postag and not str(postag).startswith("v"):
        return ""

    lemma_n = normalize_greek_lemma(lemma)
    if not lemma_n:
        return ""
    if lemma_n.endswith("μαι"):
        return "deponent"
    if lemma_n.endswith("μι"):
        return "mi"
    if lemma_n.endswith("ω"):
        return "w"
    return "irregular"


# Apply the functions to create syllabus and lesson POS category columns
combined_df['syllabus'] = combined_df['postag'].apply(parse_postag)
combined_df['pos_category'] = combined_df['postag'].apply(parse_pos_category)
combined_df['verb_subcategory'] = combined_df.apply(
    lambda row: parse_verb_subcategory(row['lemma'], row['postag']) if row['pos_category'] == 'verb' else '',
    axis=1,
)

#combined_df.sample(10)

In [50]:
def normalize_frequency_row_name(label):
    # Normalize syllabus row names to the lesson filename style.
    if not isinstance(label, str):
        return label

    normalized = label.strip().lower()
    normalized = normalized.replace(', ', '_').replace(',', '_')
    normalized = normalized.replace('/', '_').replace(' ', '_')
    normalized = normalized.replace('(', '').replace(')', '')
    normalized = re.sub(r'_+', '_', normalized).strip('_')
    return normalized

# Frequency table used for exercises (verbs split by subcategory in the same syllabus column)
verb_mask = (
    combined_df['pos_category'].eq('verb')
    & combined_df['verb_subcategory'].notna()
    & combined_df['verb_subcategory'].astype(str).ne('')
)

syllabus_with_verb_bucket = combined_df['syllabus'].where(
    ~verb_mask,
    combined_df['syllabus'] + " (" + combined_df['verb_subcategory'] + ")"
)

frequency_syllabus = (
    pd.DataFrame({
        'syllabus': syllabus_with_verb_bucket,
        'pos_category': combined_df['pos_category']
    })
    .groupby(['syllabus', 'pos_category'], dropna=False)
    .size()
    .reset_index(name='frequency')
    .sort_values('frequency', ascending=False, ignore_index=True)
)
frequency_syllabus['syllabus'] = frequency_syllabus['syllabus'].apply(normalize_frequency_row_name)

In [51]:
# Show the frequency syllabus and ask how many lessons to generate.
frequency_syllabus_display = frequency_syllabus[['syllabus', 'pos_category', 'frequency']].copy()
frequency_syllabus_display = frequency_syllabus_display.reset_index(drop=True)

print(f"Frequency syllabus rows: {len(frequency_syllabus_display)}")
print(frequency_syllabus_display.to_string(index=False))
print("")

while True:
    lesson_count_input = input(f"How many lessons should be generated? Enter a number from 1 to {len(frequency_syllabus_display)}: ").strip()
    try:
        LESSON_COUNT = int(lesson_count_input)
        if 1 <= LESSON_COUNT <= len(frequency_syllabus_display):
            break
    except ValueError:
        pass
    print(f"Please enter a whole number between 1 and {len(frequency_syllabus_display)}.")

print(f"Selection saved to LESSON_COUNT: {LESSON_COUNT}")


Frequency syllabus rows: 333
                                         syllabus   pos_category  frequency
                                               na          other      62726
                                       accusative noun/adjective      47293
                                           adverb         adverb      35486
                                       nominative noun/adjective      34935
                                         particle       particle      34846
                                          pronoun        pronoun      26999
                                         genitive noun/adjective      24873
                                      preposition    preposition      20054
                                      conjunction    conjunction      19818
                                           dative noun/adjective      19625
                                          article        article      16294
                       aorist_indicative_active_w          

## Putting sentences back together for generating exercises

In [52]:
# Count lemma frequency only for Greek lemmas, ignoring punctuation
GREEK_RE = re.compile(r'[\u0370-\u03FF\u1F00-\u1FFF]')
NON_GREEK_LETTER_RE = re.compile(r'[^\u0370-\u03FF\u1F00-\u1FFF]+')

def normalize_lemma_for_freq(lemma):
    if not isinstance(lemma, str):
        return None
    cleaned = NON_GREEK_LETTER_RE.sub('', lemma)
    return cleaned or None

def is_greek_lemma(lemma):
    return isinstance(lemma, str) and bool(GREEK_RE.search(lemma))

combined_df['lemma_clean'] = combined_df['lemma'].apply(normalize_lemma_for_freq)

lemma_counts = combined_df.loc[
    combined_df['lemma_clean'].apply(is_greek_lemma), 'lemma_clean'
].value_counts()

combined_df['lemma_frequency'] = combined_df['lemma_clean'].map(lemma_counts).astype('Int64')

combined_df[['lemma', 'lemma_clean', 'lemma_frequency']].head(20)


,lemma,lemma_clean,lemma_frequency
0,̓́Ανθρωπος,Ανθρωπος,1
1,εἰμί,εἰμί,4660
2,ἐγώ,ἐγώ,4515
3,Κυρηναῖος,Κυρηναῖος,6
4,δοκέω,δοκέω,259
5,",",None,<NA>
6,κατά,κατά,1470
7,ὁ,ὁ,21880
8,Ἄλεξις,Ἄλεξις,22
9,Τυνδάρεως,Τυνδάρεως,1


In [53]:
def assemble_sentences(df):
    attach_to_prev = {",", ".", ";", ":", "!", "?", ")", "']", "—"}
    openers = {"(", "[", "'"}

    def join_forms(forms):
        words = []
        for form in forms:
            form = str(form)
            if form in attach_to_prev and words:
                words[-1] += form
            else:
                words.append(form)

        text = " ".join(words)
        text = re.sub(r"\s+([,.:;!?—\)])", r"\1", text)
        text = re.sub(r"([\(\[])\s+", r"\1", text)
        text = re.sub(r"[\[\]\d]", "", text)
        return re.sub(r"\s+", " ", text).strip()

    rows = []
    for sent_id, group in df.groupby("sentence_id", sort=False):
        group = group.sort_values("token_index" if "token_index" in group.columns else "word_id")
        first = group.iloc[0]

        rows.append({
            "sentence_id": sent_id,
            "document_id": first["document_id"],
            "subdoc": first["subdoc"],
            "file": first["file"],
            "sentence_text": join_forms(group["form"].tolist()),
            "word_count": len(group),
        })

    return pd.DataFrame(rows)


sentences_df = assemble_sentences(combined_df)
sentences_df["sentence_index"] = range(len(sentences_df))
combined_df["sentence_index"] = combined_df.groupby("sentence_id", sort=False).ngroup()
sentences_df.sample(5)


,sentence_id,document_id,subdoc,file,sentence_text,word_count,sentence_index
834,tb_835,urn:cts:greekLit:tlg0008.tlg001.perseus-grc1,13.81,tlg0008.tlg001.perseus-grc1.13.tb.xml,καὶ πρὸς τόδε ἠμείφθη ὁ Ἐρετριεὺς ἢ Ἐρυθραῖος ...,99,834
3350,tb_3351,urn:cts:greekLit:tlg0012.tlg002.perseus-grc1,8.535,tlg0012.tlg002.perseus-grc1.tb.xml,"αἶψα δὲ Φαιήκεσσι φιληρέτμοισι μετηύδα · "" κέκ...",48,3350
3159,tb_3160,urn:cts:greekLit:tlg0012.tlg002.perseus-grc1,7.239,tlg0012.tlg002.perseus-grc1.tb.xml,"οὐ δὴ φῆς ἐπὶ πόντον ἀλώμενος ἐνθάδ̓ ἱκέσθαι;""...",98,3159
2662,tb_2663,urn:cts:greekLit:tlg0012.tlg002.perseus-grc1,4.328-4.331,tlg0012.tlg002.perseus-grc1.tb.xml,""" τὸν δὲ μέγ̓ ὀχθήσας προσέθη ξανθὸς Μενέλαος ...",58,2662
1474,tb_1475,urn:cts:greekLit:tlg0012.tlg002.perseus-grc1,18.333,tlg0012.tlg002.perseus-grc1.tb.xml,"ἦ ἀλύεις, ὅτι Ἶρον ἐνίκησας τὸν ἀλήτην; οἷον δ...",44,1474


## Scoring Sentences: Lemma Frequency & Length

In [54]:
def add_sentence_scores(sentences_df, combined_df):
    for col in ("sentence_id", "word_count"):
        if col not in sentences_df.columns:
            raise ValueError(f"sentences_df missing column: {col}")
    for col in ("sentence_id", "lemma"):
        if col not in combined_df.columns:
            raise ValueError(f"combined_df missing column: {col}")

    out = sentences_df.copy()
    greek = combined_df[combined_df["lemma"].apply(is_greek_lemma)].copy()

    # Reuse precomputed lemma_frequency if available; otherwise compute locally
    if "lemma_frequency" in greek.columns:
        greek["lemma_frequency"] = pd.to_numeric(greek["lemma_frequency"], errors="coerce").fillna(0.0)
    else:
        counts = greek["lemma"].value_counts()
        greek["lemma_frequency"] = greek["lemma"].map(counts).astype(float)

    sent_avg = (
        greek.groupby("sentence_id", as_index=False)["lemma_frequency"]
        .mean()
        .rename(columns={"lemma_frequency": "avg_lemma_freq"})
    )

    out = out.drop(columns=["avg_lemma_freq"], errors="ignore").merge(sent_avg, on="sentence_id", how="left")
    out["avg_lemma_freq"] = out["avg_lemma_freq"].fillna(0.0)

    def to_0_100(s):
        m = s.max()
        return (s / m * 100) if pd.notna(m) and m > 0 else pd.Series(0.0, index=s.index)

    out["lemma_frequency_score"] = to_0_100(out["avg_lemma_freq"])
    out["lemma_difficulty_score"] = 100 - out["lemma_frequency_score"]
    out["sentence_length_score"] = to_0_100(out["word_count"])
    out["difficulty_score"] = (out["lemma_difficulty_score"] + 2 * out["sentence_length_score"]) / 3

    return out

sentences_df = add_sentence_scores(sentences_df, combined_df)
sentences_df.sort_values("difficulty_score").head(10)


,sentence_id,document_id,subdoc,file,sentence_text,word_count,sentence_index,avg_lemma_freq,lemma_frequency_score,lemma_difficulty_score,sentence_length_score,difficulty_score
3696,tb_3697,urn:cts:greekLit:tlg0012.tlg001.perseus-grc1,9.167,tlg0012.tlg001.perseus-grc1.tb.xml,εἰ δ̓ ἄγε τοὺς ἂν ἐγὼ ἐπιόψομαι οἳ δὲ πιθέσθων.,11,3696,8849.1,100.0,0.0,1.476510,0.98434
3712,tb_3713,urn:cts:greekLit:tlg0012.tlg001.perseus-grc1,9.206-9.208,tlg0012.tlg001.perseus-grc1.tb.xml,"τῷ δ̓ ἔχεν Αὐτομέδων, τάμνεν δ̓ ἄρα δῖος Ἀχιλλ...",13,3712,6954.666667,78.591797,21.408203,1.744966,8.299379
3673,tb_3674,urn:cts:greekLit:tlg0012.tlg001.perseus-grc1,9.79,tlg0012.tlg001.perseus-grc1.tb.xml,"ὣς ἔφαθ̓, οἳ δ̓ ἄρα τοῦ μάλα μὲν κλύον ἠδὲ πίθ...",13,3673,6694.818182,75.655357,24.344643,1.744966,9.278192
2528,tb_2529,urn:cts:greekLit:tlg0012.tlg002.perseus-grc1,3.384,tlg0012.tlg002.perseus-grc1.tb.xml,""" ὣς ἔφατ̓ εὐχόμενος, τοῦ δ̓ ἔκλυε Παλλὰς Ἀθήν...",31,2528,6653.458333,75.187966,24.812034,4.161074,11.044727
3755,tb_3756,urn:cts:greekLit:tlg0012.tlg001.perseus-grc1,9.378,tlg0012.tlg001.perseus-grc1.tb.xml,"ἐχθρὰ δέ μοι τοῦ δῶρα, τίω δέ μιν ἐν καρὸς αἴσῃ.",13,3755,6109.818182,69.044515,30.955485,1.744966,11.481806
3331,tb_3332,urn:cts:greekLit:tlg0012.tlg002.perseus-grc1,8.461-8.462,tlg0012.tlg002.perseus-grc1.tb.xml,""" τὴν δ̓ ἀπαμειβόμενος προσέφη πολύμητις Ὀδυσσ...",17,3331,6148.692308,69.483815,30.516185,2.281879,11.693314
3699,tb_3700,urn:cts:greekLit:tlg0012.tlg001.perseus-grc1,9.173,tlg0012.tlg001.perseus-grc1.tb.xml,"ὣς φάτο, τοῖσι δὲ πᾶσιν ἑαδότα μῦθον ἔειπεν.",10,3699,5963.0,67.385384,32.614616,1.342282,11.766393
3816,tb_3817,urn:cts:greekLit:tlg0012.tlg001.perseus-grc1,9.666-9.668,tlg0012.tlg001.perseus-grc1.tb.xml,οἳ δ̓ ὅτε δὴ κλισίῃσιν ἐν Ἀτρεΐδαο γένοντο.,11,3816,5866.25,66.292052,33.707948,1.476510,12.220323
3627,tb_3628,urn:cts:greekLit:tlg0012.tlg001.perseus-grc1,8.482-8.483,tlg0012.tlg001.perseus-grc1.tb.xml,"ὣς φάτο, τὸν δ̓ οὔ τι προσέφη λευκώλενος Ἥρη.",12,3627,5761.222222,65.105177,34.894823,1.610738,12.705433
1669,tb_1670,urn:cts:greekLit:tlg0012.tlg002.perseus-grc1,2.8,tlg0012.tlg002.perseus-grc1.tb.xml,"οἱ μὲν ἐκήρυσσον, τοὶ δ̓ ἠγείροντο μάλ̓ ὦκα. ο...",27,1669,5966.521739,67.425182,32.574818,3.624161,13.27438


# Exercises

In [55]:
def split_syllabus_label_and_bucket(syllabus_label):
    if not isinstance(syllabus_label, str):
        return syllabus_label, None
    m = re.match(r'^(.*)\s\(([^()]*)\)$', syllabus_label.strip())
    if not m:
        return syllabus_label, None
    return m.group(1), m.group(2)


SIMPLE_POS_LESSONS = {
    'adverb': 'd',
    'preposition': 'r',
    'particle': 'g',
    'conjunction': 'c',
    'interjection': 'i',
}

POS_EXERCISE_PROMPTS = {
    'verb': {
        1: 'What is the number and person of the following verbs?',
        2: 'What is the number and person of the marked verbs in each sentence?',
        3: 'Find one finite verb in each sentence and give its dictionary lemma.',
    },
    'noun/adjective': {
        1: 'Which declension do the following words belong to?',
        2: 'What is the gender, case, and number of the marked words?',
        3: 'Translate each sentence and identify the marked noun or adjective.',
    },
    'article': {
        1: 'What is the gender, case, and number of the marked words?',
        2: 'What is the gender, case, and number of the marked articles?',
        3: 'Identify the article in each sentence and give its gender, case, and number.',
    },
    'pronoun': {
        1: 'What is the gender, case, and number of the following pronouns?',
        2: 'What is the gender, case, and number of the marked pronouns?',
        3: 'Identify each pronoun and state its gender, case, and number.',
    },
    'adverb': {
        1: 'What are the following adverbs?',
        2: 'Identify the marked adverb in each sentence and note its role.',
        3: 'Translate each sentence and explain how the adverb affects the meaning.',
    },
    'preposition': {
        1: 'What are the following prepositions?',
        2: 'Identify the marked preposition in each sentence and give the case it governs.',
        3: 'State the governed case for each marked preposition and identify its complement.',
    },
    'particle': {
        1: 'What are the following particles?',
        2: 'Identify the marked particle in each sentence and explain its effect.',
        3: 'Translate each sentence and note the contribution of the particle.',
    },
    'conjunction': {
        1: 'What are the following conjunctions?',
        2: 'Identify the marked conjunction in each sentence and explain what it connects.',
        3: 'Translate each sentence and describe the clause or phrase joined by the conjunction.',
    },
    'interjection': {
        1: 'What are the following interjections?',
        2: 'Identify the marked interjection in each sentence and give its force or meaning.',
        3: 'Translate each sentence and explain the interjection.',
    },
}


def get_exercise_prompt(lesson_pos_category, set_number):
    return POS_EXERCISE_PROMPTS.get(lesson_pos_category, POS_EXERCISE_PROMPTS['noun/adjective']).get(
        set_number,
        'Translate each sentence.',
    )


def decode_nominal_features(postag):
    if not isinstance(postag, str) or len(postag) < 8:
        return 'gender: unknown; number: unknown; case: unknown'

    gender_map = {'m': 'masculine', 'f': 'feminine', 'n': 'neuter', 'c': 'common'}
    number_map = {'s': 'singular', 'p': 'plural', 'd': 'dual'}
    case_map = {'n': 'nominative', 'g': 'genitive', 'd': 'dative', 'a': 'accusative', 'v': 'vocative'}

    gender = gender_map.get(postag[6], 'unknown')
    number = number_map.get(postag[2], 'unknown')
    case_value = case_map.get(postag[7], 'unknown')
    return f'gender: {gender}; number: {number}; case: {case_value}'


def infer_preposition_governed_case(sentence_rows, preposition_token_index):
    if sentence_rows is None or sentence_rows.empty or preposition_token_index is None:
        return 'unknown'

    order_column = 'token_index' if 'token_index' in sentence_rows.columns else 'word_id'
    ordered_rows = sentence_rows.sort_values(order_column)
    tail_rows = ordered_rows[ordered_rows[order_column] > preposition_token_index]

    for _, candidate in tail_rows.iterrows():
        postag = candidate.get('postag')
        if not isinstance(postag, str) or len(postag) < 8:
            continue
        if postag[0] in {'n', 'a', 'p'}:
            return decode_nominal_features(postag).split(';')[-1].split(':', 1)[-1].strip()
        if str(candidate.get('form', '')).strip() in {'.', '?', ';', '!', ':'}:
            break

    return 'unknown'


def describe_topic_word(row, lesson_pos_category, sentence_rows=None):
    lemma = row.get('lemma', '')
    postag = row.get('postag', '')

    if lesson_pos_category == 'verb':
        return decode_set1_verb_features(postag)
    if lesson_pos_category in {'noun/adjective', 'article', 'pronoun'}:
        return decode_nominal_features(postag)
    if lesson_pos_category == 'preposition':
        governed_case = infer_preposition_governed_case(sentence_rows, row.get('token_index'))
        return f'preposition; governs case: {governed_case}'
    if lesson_pos_category in {'adverb', 'particle', 'conjunction', 'interjection'}:
        return f'{lesson_pos_category}; lemma: {lemma}'
    return f'lemma: {lemma}'


def get_topic_rows_for_label(syllabus_label, combined_df):
    base_label, verb_bucket = split_syllabus_label_and_bucket(syllabus_label)
    if verb_bucket is None:
        return combined_df[combined_df['syllabus'] == syllabus_label].copy()

    rows = combined_df[combined_df['syllabus'] == base_label].copy()
    rows = rows[(rows['pos_category'] == 'verb') & (rows['verb_subcategory'] == verb_bucket)]
    return rows


def get_topic_sentences(syllabus_label, combined_df, sentences_df, num_sentences=40):
    matching_rows = get_topic_rows_for_label(syllabus_label, combined_df)
    if matching_rows.empty:
        return pd.DataFrame()

    matching_sentence_indices = set(matching_rows['sentence_index'].unique())
    if not matching_sentence_indices:
        return pd.DataFrame()

    topic_sentences = sentences_df[sentences_df['sentence_index'].isin(matching_sentence_indices)].copy()
    if topic_sentences.empty:
        return pd.DataFrame()

    return topic_sentences.sort_values('difficulty_score').head(num_sentences)


def filter_topic_rows_by_lesson_rules(syllabus_label, lesson_pos_category, topic_rows):
    case_lessons = {'accusative', 'dative', 'genitive', 'nominative', 'vocative'}
    if syllabus_label == 'article':
        return topic_rows[topic_rows['postag'].str.startswith('l', na=False)]
    if syllabus_label in case_lessons:
        return topic_rows[topic_rows['postag'].str.startswith(('n', 'a'), na=False)]
    if lesson_pos_category == 'verb':
        return topic_rows[topic_rows['postag'].str.startswith('v', na=False)]
    if lesson_pos_category == 'pronoun':
        return topic_rows[topic_rows['postag'].str.startswith('p', na=False)]
    if lesson_pos_category in SIMPLE_POS_LESSONS:
        prefix = SIMPLE_POS_LESSONS[lesson_pos_category]
        return topic_rows[topic_rows['postag'].str.startswith(prefix, na=False)]
    return topic_rows


def mark_topic_words_in_sentence(sentence_text, target_forms):
    if not target_forms:
        return sentence_text

    marked_text = sentence_text
    for form in sorted(target_forms, key=len, reverse=True):
        if not form:
            continue
        marked_text = re.sub(rf'(?<!\w)({re.escape(form)})(?!\w)', r'<u>\1</u>', marked_text)
    return marked_text


def get_topic_words(syllabus_label, lesson_pos_category, combined_df, num_words=15):
    topic_rows = get_topic_rows_for_label(syllabus_label, combined_df)
    if topic_rows.empty:
        return pd.DataFrame()

    topic_rows = topic_rows.dropna(subset=['form', 'lemma', 'postag']).copy()
    topic_rows['form'] = topic_rows['form'].astype(str).str.strip()
    topic_rows['lemma'] = topic_rows['lemma'].astype(str).str.strip()
    topic_rows['postag'] = topic_rows['postag'].astype(str).str.strip()
    topic_rows = topic_rows[(topic_rows['form'] != '') & (topic_rows['lemma'] != '') & (topic_rows['postag'] != '')]
    topic_rows = topic_rows[topic_rows['lemma'].apply(is_greek_lemma)]

    if topic_rows.empty:
        return pd.DataFrame()

    topic_rows = filter_topic_rows_by_lesson_rules(syllabus_label, lesson_pos_category, topic_rows)
    if topic_rows.empty:
        return pd.DataFrame()

    if 'lemma_frequency' not in topic_rows.columns:
        local_counts = topic_rows['lemma'].value_counts()
        topic_rows['lemma_frequency'] = topic_rows['lemma'].map(local_counts)

    topic_rows['lemma_frequency'] = pd.to_numeric(topic_rows['lemma_frequency'], errors='coerce').fillna(0)
    topic_rows = topic_rows.sort_values('lemma_frequency', ascending=False)

    topic_words = topic_rows.drop_duplicates(subset=['lemma'], keep='first').head(num_words)
    return topic_words[['form', 'lemma', 'lemma_frequency']]


def generate_exercises_for_topic(
    syllabus_label,
    lesson_pos_category,
    combined_df,
    sentences_df,
    num_sentences=40,
    sentences_per_exercise=10,
):
    # Build exercises for one syllabus topic.
    exercise_blocks = []

    topic_words = get_topic_words(syllabus_label, lesson_pos_category, combined_df, num_words=15)
    exercise_set1 = format_exercise_set1(topic_words, lesson_pos_category)
    if exercise_set1:
        exercise_blocks.append(exercise_set1)

    topic_sentences = get_topic_sentences(
        syllabus_label,
        combined_df,
        sentences_df,
        num_sentences=num_sentences,
    )

    if not topic_sentences.empty:
        topic_rows = get_topic_rows_for_label(syllabus_label, combined_df)
        topic_rows = topic_rows.dropna(subset=['form', 'postag'])
        topic_rows['form'] = topic_rows['form'].astype(str).str.strip()
        topic_rows['postag'] = topic_rows['postag'].astype(str).str.strip()
        topic_rows = topic_rows[(topic_rows['form'] != '') & (topic_rows['postag'] != '')]
        topic_rows = filter_topic_rows_by_lesson_rules(syllabus_label, lesson_pos_category, topic_rows)

        sentence_form_lookup = (
            topic_rows.groupby('sentence_index')['form']
            .apply(lambda s: set(s.tolist()))
            .to_dict()
            if not topic_rows.empty else {}
        )

        topic_sentences = topic_sentences.copy()
        topic_sentences['sentence_text_marked'] = topic_sentences.apply(
            lambda row: mark_topic_words_in_sentence(
                row['sentence_text'],
                sentence_form_lookup.get(row['sentence_index'], set())
            ),
            axis=1,
        )

    required_total = 2 * sentences_per_exercise
    if len(topic_sentences) >= required_total:
        exercise_set2 = format_exercise_set2(topic_sentences.iloc[0:sentences_per_exercise], lesson_pos_category)
        if lesson_pos_category == 'verb':
            exercise_set3 = format_verb_exercise_set3(topic_sentences.iloc[sentences_per_exercise:required_total])
        else:
            exercise_set3 = format_exercise_set3(topic_sentences.iloc[sentences_per_exercise:required_total], lesson_pos_category)

        exercise_blocks.extend([exercise_set2, exercise_set3])

    return "\n".join(exercise_blocks)

In [56]:
# Support normalized syllabus labels (e.g., present_participle_active_w) in exercise/topic lookup.
# Cache the normalized series to avoid recalculating it repeatedly
_normalized_syllabus_cache = None

def get_topic_rows_for_label(syllabus_label, combined_df):
    global _normalized_syllabus_cache
    
    # First try legacy label parsing.
    base_label, verb_bucket = split_syllabus_label_and_bucket(syllabus_label)
    if verb_bucket is None:
        direct = combined_df[combined_df['syllabus'] == syllabus_label].copy()
        if not direct.empty:
            return direct
    else:
        direct = combined_df[combined_df['syllabus'] == base_label].copy()
        if not direct.empty:
            return direct[(direct['pos_category'] == 'verb') & (direct['verb_subcategory'] == verb_bucket)]

    # Fallback: match normalized labels against normalized corpus syllabus.
    normalized_target = normalize_frequency_row_name(syllabus_label)
    
    # Compute normalized series only once and cache it
    if _normalized_syllabus_cache is None or len(_normalized_syllabus_cache) != len(combined_df):
        _normalized_syllabus_cache = combined_df['syllabus'].apply(normalize_frequency_row_name)
    
    normalized_series = _normalized_syllabus_cache

    # Detect normalized verb bucket suffixes.
    verb_suffix_map = {
        '_w': 'w',
        '_mi': 'mi',
        '_deponent': 'deponent',
        '_irregular': 'irregular',
    }

    for suffix, raw_bucket in verb_suffix_map.items():
        if normalized_target.endswith(suffix):
            base_norm = normalized_target[:-len(suffix)]
            rows = combined_df[
                (normalized_series == base_norm)
                & (combined_df['pos_category'] == 'verb')
                & (combined_df['verb_subcategory'].isin([raw_bucket, suffix.lstrip('_')]))
            ].copy()
            return rows

    # Non-verb normalized match.
    return combined_df[normalized_series == normalized_target].copy()

print('Topic lookup updated for normalized syllabus labels with caching.')

Topic lookup updated for normalized syllabus labels with caching.


In [57]:
def format_exercise_set2(exercise_sentences, lesson_pos_category):
    # Exercise Set 2 changes its prompt based on the lesson POS category.
    prompt = get_exercise_prompt(lesson_pos_category, 2)

    lines = ["### Exercise Set 2", "", prompt, ""]
    for idx, (_, row) in enumerate(exercise_sentences.iterrows(), 1):
        sentence_text = row.get('sentence_text_marked', row['sentence_text'])
        lines.append(f"{idx}. {sentence_text}")
        lines.append("")
    return "\n".join(lines)

In [58]:
def format_exercise_set3(exercise_sentences, lesson_pos_category):
    # Exercise Set 3 changes its content by part of speech.
    if exercise_sentences is None or exercise_sentences.empty:
        return ""

    prompt = get_exercise_prompt(lesson_pos_category, 3)
    lines = ["### Exercise Set 3", "", prompt, ""]
    for idx, (_, row) in enumerate(exercise_sentences.iterrows(), 1):
        sentence_text = row.get('sentence_text_marked', row['sentence_text'])
        lines.append(f"{idx}. {sentence_text}")
        lines.append("")
    return "\n".join(lines)

In [59]:
def format_exercise_set1(topic_words, lesson_pos_category):
    # Exercise Set 1 changes its prompt based on the lesson POS category.
    if topic_words is None or topic_words.empty:
        return ""

    prompt = get_exercise_prompt(lesson_pos_category, 1)

    lines = ["### Exercise Set 1", "", prompt, ""]
    for idx, (_, row) in enumerate(topic_words.iterrows(), 1):
        lines.append(f"{idx}. {row['form']} (lemma: {row['lemma']})")
    lines.append("")
    return "\n".join(lines)


def format_verb_exercise_set2(exercise_sentences):
    return format_exercise_set2(exercise_sentences, 'verb')


def format_verb_exercise_set3(exercise_sentences):
    lines = [
        "### Exercise Set 3 (Verbs)",
        "",
        "Find one finite verb in each sentence and give its dictionary lemma.",
        "",
    ]
    for idx, (_, row) in enumerate(exercise_sentences.iterrows(), 1):
        sentence_text = row.get('sentence_text_marked', row['sentence_text'])
        lines.append(f"{idx}. {sentence_text}")
        lines.append("")
    return "\n".join(lines)

In [60]:
def get_topic_words(syllabus_label, lesson_pos_category, combined_df, num_words=15):
    # Return the most frequent-topic word forms for one syllabus label.
    topic_rows = get_topic_rows_for_label(syllabus_label, combined_df)
    if topic_rows.empty:
        return pd.DataFrame()

    topic_rows = topic_rows.dropna(subset=['form', 'lemma', 'postag']).copy()
    topic_rows['form'] = topic_rows['form'].astype(str).str.strip()
    topic_rows['lemma'] = topic_rows['lemma'].astype(str).str.strip()
    topic_rows['postag'] = topic_rows['postag'].astype(str).str.strip()
    topic_rows = topic_rows[(topic_rows['form'] != '') & (topic_rows['lemma'] != '') & (topic_rows['postag'] != '')]
    topic_rows = topic_rows[topic_rows['lemma'].apply(is_greek_lemma)]

    if topic_rows.empty:
        return pd.DataFrame()

    topic_rows = filter_topic_rows_by_lesson_rules(syllabus_label, lesson_pos_category, topic_rows)
    if topic_rows.empty:
        return pd.DataFrame()

    if 'lemma_frequency' not in topic_rows.columns:
        local_counts = topic_rows['lemma'].value_counts()
        topic_rows['lemma_frequency'] = topic_rows['lemma'].map(local_counts)

    topic_rows['lemma_frequency'] = pd.to_numeric(topic_rows['lemma_frequency'], errors='coerce').fillna(0)
    topic_rows = topic_rows.sort_values('lemma_frequency', ascending=False)

    topic_words = topic_rows.drop_duplicates(subset=['lemma'], keep='first').head(num_words)
    return topic_words[['form', 'lemma', 'lemma_frequency']]

# Generate Frequency-Based textbook in Markdown

In [61]:
def syllabus_to_filename(syllabus_label):
    # Convert a frequency_syllabus label to a lesson module filename.
    if pd.isna(syllabus_label) or syllabus_label == 'NA':
        return None
    # Replace commas, slashes, and spaces with underscores, convert to lowercase
    filename = syllabus_label.replace(', ', '_').replace('/', '_').replace(' ', '_').lower() + '.md'
    return filename


def decode_set1_article_features(postag):
    # Decode article postag into gender, number, case.
    return decode_nominal_features(postag)


def decode_set1_verb_features(postag):
    # Decode and explain verb postag in a readable way.
    if not isinstance(postag, str) or len(postag) < 9:
        return "POS tag explanation unavailable"

    person_map = {
        '1': '1st person',
        '2': '2nd person',
        '3': '3rd person',
        '-': 'not marked',
    }
    number_map = {
        's': 'singular',
        'p': 'plural',
        'd': 'dual',
        '-': 'not marked',
    }
    tense_map_local = {
        'p': 'present',
        'i': 'imperfect',
        'f': 'future',
        'a': 'aorist',
        'r': 'perfect',
        'l': 'pluperfect',
        't': 'future perfect',
        '-': 'not marked',
    }
    mood_map_local = {
        'i': 'indicative',
        's': 'subjunctive',
        'o': 'optative',
        'm': 'imperative',
        'n': 'infinitive',
        'p': 'participle',
        '-': 'not marked',
    }
    voice_map_local = {
        'a': 'active',
        'm': 'middle',
        'p': 'passive',
        'e': 'middle/passive',
        '-': 'not marked',
    }

    person_code = postag[1]
    number_code = postag[2]
    tense_code = postag[3]
    mood_code = postag[4]
    voice_code = postag[5]

    person_value = person_map.get(person_code, 'unknown')
    number_value = number_map.get(number_code, 'unknown')
    tense_value = tense_map_local.get(tense_code, 'unknown')
    mood_value = mood_map_local.get(mood_code, 'unknown')
    voice_value = voice_map_local.get(voice_code, 'unknown')

    return (
        f"postag {postag} -> "
        f"person: {person_value} ({person_code}); "
        f"number: {number_value} ({number_code}); "
        f"tense: {tense_value} ({tense_code}); "
        f"mood: {mood_value} ({mood_code}); "
        f"voice: {voice_value} ({voice_code})"
    )


def build_exercise_set1_answer_key(syllabus_label, lesson_pos_category, combined_df, num_words=15):
    # Build Set 1 answer key from treebank data for the current lesson.
    if combined_df is None:
        return ""

    topic_words = get_topic_words(syllabus_label, lesson_pos_category, combined_df, num_words=num_words)
    lines = ["#### Answer Key for Exercise Set 1", ""]

    if topic_words is None or topic_words.empty:
        lines.append("*No answer key available for Exercise Set 1.*")
        lines.append("")
        return '\n'.join(lines)

    for idx, (_, row) in enumerate(topic_words.iterrows(), 1):
        sentence_rows = None
        if 'sentence_index' in combined_df.columns and pd.notna(row.get('sentence_index')):
            sentence_rows = combined_df[combined_df['sentence_index'] == row['sentence_index']].copy()

        answer = describe_topic_word(row, lesson_pos_category, sentence_rows=sentence_rows)
        lines.append(f"{idx}. {row['form']} (lemma: {row['lemma']}) -> {answer}")

    lines.append("")
    return '\n'.join(lines)


def generate_syllabus_markdown_with_content(frequency_syllabus, grammar_folder, combined_df=None, sentences_df=None):
    # Generate a markdown file with table of contents, lesson content, exercises, and Set 1 answer key.
    markdown_content = []
    markdown_content.append("# A Frequency-Based Textbook for Ancient Greek Grammar")
    markdown_content.append("")
    markdown_content.append("This syllabus organizes grammar lessons by frequency of occurrence in the text.")
    markdown_content.append("")

    # Build table of contents
    markdown_content.append("## Table of Contents")
    markdown_content.append("")

    lesson_count = 0
    lesson_data = []  # Store data for second pass

    lesson_limit = int(globals().get('LESSON_COUNT', 20) or 20)
    lesson_rows = frequency_syllabus[frequency_syllabus['syllabus'].notna() & (frequency_syllabus['syllabus'] != 'NA')].head(lesson_limit)

    for idx, row in lesson_rows.iterrows():
        syllabus_label = row['syllabus']
        frequency = row['frequency']
        pos_category = row.get('pos_category', 'other')

        lesson_count += 1
        filename = syllabus_to_filename(syllabus_label)
        lesson_path = grammar_folder / filename

        lesson_data.append({
            'rank': lesson_count,
            'label': syllabus_label,
            'frequency': frequency,
            'pos_category': pos_category,
            'filename': filename,
            'path': lesson_path
        })

        markdown_content.append(f"{lesson_count}. [{syllabus_label}](#{filename.replace('.md', '')})")

    markdown_content.append("")
    markdown_content.append("---")
    markdown_content.append("")

    about_path = Path(r"C:\Users\Farnoosh\Documents\GitHub\didaskon\homer-textbook\lessons-no-decl\about.md")
    if about_path.exists():
        try:
            with open(about_path, 'r', encoding='utf-8') as f:
                markdown_content.append(f.read())
        except Exception as e:
            markdown_content.append(f"*Error reading about file: {str(e)}*")
        markdown_content.append("")
        markdown_content.append("---")
        markdown_content.append("")

    for lesson in lesson_data:
        markdown_content.append(f"## {lesson['rank']}. {lesson['label']}")
        markdown_content.append(f"**Part of Speech Family:** {lesson['pos_category']}")
        markdown_content.append(f"**Frequency:** {lesson['frequency']}")
        markdown_content.append("")

        if lesson['path'].exists():
            try:
                with open(lesson['path'], 'r', encoding='utf-8') as f:
                    lesson_content = f.read()
                    markdown_content.append(lesson_content)
            except Exception as e:
                markdown_content.append(f"*Error reading file: {str(e)}*")
        else:
            markdown_content.append(f"*Module file not yet created: {lesson['filename']}*")

        markdown_content.append("")
        markdown_content.append("### Exercises")
        markdown_content.append("")

        if combined_df is not None and sentences_df is not None:
            exercises = generate_exercises_for_topic(
                lesson['label'],
                lesson['pos_category'],
                combined_df,
                sentences_df,
            )
            if exercises:
                markdown_content.append(exercises)
            else:
                markdown_content.append(f"*No exercises available for {lesson['label']}.*")
        else:
            markdown_content.append("*Exercises will be added here later.*")

        if combined_df is not None:
            markdown_content.append("")
            answer_key = build_exercise_set1_answer_key(
                lesson['label'],
                lesson['pos_category'],
                combined_df,
                num_words=15,
            )
            if answer_key:
                markdown_content.append(answer_key)

        markdown_content.append("")
        markdown_content.append("---")
        markdown_content.append("")

    return '\n'.join(markdown_content)


# Generate the syllabus markdown with full content including exercises
syllabus_markdown = generate_syllabus_markdown_with_content(frequency_syllabus, GRAMMAR_FOLDER, combined_df, sentences_df)

# Write to file
output_file = Path("textbook.md")
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(syllabus_markdown)

print(f"Textbook markdown with lesson content and exercises generated and saved to {output_file}")
print(f"Total lessons: {len([x for x in frequency_syllabus.head(int(globals().get('LESSON_COUNT', 20) or 20))['syllabus'] if x != 'NA' and pd.notna(x)])}")


Textbook markdown with lesson content and exercises generated and saved to textbook.md
Total lessons: 40


In [62]:
# Regenerate textbook markdown using the updated generator
syllabus_markdown = generate_syllabus_markdown_with_content(frequency_syllabus, GRAMMAR_FOLDER, combined_df, sentences_df)

# Write to file
output_file = Path("textbook.md")
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(syllabus_markdown)

lesson_total = int(len([row for _, row in frequency_syllabus[frequency_syllabus['syllabus'].notna() & (frequency_syllabus['syllabus'] != 'NA')].head(int(globals().get('LESSON_COUNT', 20) or 20)).iterrows()]))
print(f"Textbook markdown with lesson content and exercises generated and saved to {output_file}")
print(f"Total lessons (selected): {lesson_total}")

Textbook markdown with lesson content and exercises generated and saved to textbook.md
Total lessons (selected): 40


In [63]:
from pathlib import Path
from markdown import markdown as md_to_html

def markdown_to_html(markdown_file, doc_title=None):
    markdown_path = Path(markdown_file)
    if not markdown_path.exists():
        raise FileNotFoundError(f"Markdown file not found: {markdown_file}")

    html_path = markdown_path.with_suffix(".html")
    
    markdown_content = markdown_path.read_text(encoding="utf-8")
    html_content = md_to_html(markdown_content, extensions=["extra", "toc", "tables"])

    page_title = doc_title or "Frequency-Based Grammar Textbook"

    full_html = f"""<!doctype html>
<html lang="grc">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width,initial-scale=1">
  <title>{page_title}</title>
  <style>
    @page {{ margin: 1.5cm; size: A4; }}
    * {{ box-sizing: border-box; }}
    body {{
      margin: 0; padding: 1.5cm;
      font: 11pt/1.7 "Helvetica", Arial, sans-serif;
      color: #333; background: #fff;
    }}
    h1,h2,h3 {{ page-break-after: avoid; color: #1a1a1a; margin: 1rem 0 .6rem; }}
    h1 {{ font-size: 2rem; border-bottom: 2px solid #2980b9; padding-bottom: .4rem; }}
    h2 {{ font-size: 1.35rem; border-left: 4px solid #3498db; padding-left: .6rem; }}
    h3 {{ font-size: 1.1rem; }}
    p, li {{ text-align: justify; }}
    table {{ width: 100%; border-collapse: collapse; margin: 1rem 0; page-break-inside: avoid; font-size: .9rem; }}
    th, td {{ border: 1px solid #ccc; padding: .45rem; text-align: left; }}
    th {{ background: #f2f4f6; }}
    pre {{ background: #f6f8fa; padding: .8rem; border: 1px solid #ddd; overflow-x: auto; }}
    code {{ font-family: "Courier New", monospace; }}
    a {{ color: #2980b9; text-decoration: none; }}
    a:hover {{ text-decoration: underline; }}
  </style>
</head>
<body lang="grc">
{html_content}
</body>
</html>"""

    html_path.write_text(full_html, encoding="utf-8")
    return html_path


html_output = markdown_to_html(str(output_file), doc_title=title if "title" in globals() else None)

print("Documentation created successfully!")
print(f"Primary file: {html_output}")
print(f"Size: {html_output.stat().st_size / 1024:.2f} KB")

Documentation created successfully!
Primary file: textbook.html
Size: 705.39 KB
